In [16]:
from ibapi.client import * # EClient class (used to SEND requests to TWS)
from ibapi.wrapper import * # EWrapper class (used to RECEIVE responses from TWS)
import datetime
import time
import threading
import pandas as pd

In [17]:
port = 4002 # paper trading port

In [18]:
class TestApp(EClient, EWrapper):
    def __init__(self):
        EClient.__init__(self, self)
        self.data = {}
        self.reqId_to_symbol = {}
        self.finished = set()

    def nextValidId(self, orderId: OrderId):
        self.orderId = orderId
        print(f"Connected. Next valid order ID: {orderId}")

    def nextId(self):
        self.orderId += 1
        return self.orderId

    def error(self, reqId, errorTime, errorCode, errorString, advancedOrderRejectJson=""):
        print(f"reqId: {reqId} | time: {errorTime} | code: {errorCode} | msg: {errorString}")

    def historicalData(self, reqId, bar):
        self.data[reqId].append({
            "symbol": self.reqId_to_symbol.get(reqId, "UNKNOWN"),
            "date":   bar.date,
            "open":   bar.open,
            "high":   bar.high,
            "low":    bar.low,
            "close":  bar.close,
            "volume": bar.volume
        })

    def historicalDataEnd(self, reqId, start, end):
        symbol = self.reqId_to_symbol.get(reqId, "UNKNOWN")
        print(f"Done: {symbol} | {len(self.data[reqId])} bars received.")
        self.finished.add(reqId)
        self.cancelHistoricalData(reqId)

In [19]:
# Disconnect safely if already connected (useful when re-running the cell)
try:
    app.disconnect()
    time.sleep(1)
except:
    pass

app = TestApp()
app.connect("127.0.0.1", 4002, clientId=0)

thread = threading.Thread(target=app.run, daemon=True)
thread.start()

time.sleep(1)  # wait for connection to establish

Connected. Next valid order ID: 1
reqId: -1 | time: 1775518691343 | code: 2104 | msg: Market data farm connection is OK:usfarm
reqId: -1 | time: 1775518691345 | code: 2106 | msg: HMDS data farm connection is OK:ushmds
reqId: -1 | time: 1775518691349 | code: 2158 | msg: Sec-def data farm connection is OK:secdefil


In [20]:
tickers = ["SPY","QQQ","IWM","VXUS","EEM","XLK","XLF","XLV","XLE","VNQ","TLT","GLD"]

for symbol in tickers:
    contract = Contract()
    contract.symbol   = symbol
    contract.secType  = "STK"
    contract.exchange = "SMART"
    contract.currency = "USD"

    req_id = app.nextId()
    app.data[req_id] = []
    app.reqId_to_symbol[req_id] = symbol

    app.reqHistoricalData(
        req_id,
        contract,
        "",
        "10 Y",
        "1 day",
        "TRADES",
        1,
        1,
        False,
        []
    )

    time.sleep(0.3)

while len(app.finished) < len(tickers):
    time.sleep(1)

Done: SPY | 2511 bars received.
reqId: 2 | time: 1775518694861 | code: 366 | msg: No historical data query found for ticker id:2
Done: QQQ | 2512 bars received.
reqId: 3 | time: 1775518695193 | code: 366 | msg: No historical data query found for ticker id:3
Done: IWM | 2511 bars received.
reqId: 4 | time: 1775518695514 | code: 366 | msg: No historical data query found for ticker id:4
Done: VXUS | 2511 bars received.
reqId: 5 | time: 1775518695650 | code: 366 | msg: No historical data query found for ticker id:5
Done: EEM | 2511 bars received.
reqId: 6 | time: 1775518696042 | code: 366 | msg: No historical data query found for ticker id:6
Done: XLK | 2511 bars received.
reqId: 7 | time: 1775518696328 | code: 366 | msg: No historical data query found for ticker id:7
Done: XLF | 2511 bars received.
reqId: 8 | time: 1775518696663 | code: 366 | msg: No historical data query found for ticker id:8
Done: XLV | 2511 bars received.
reqId: 9 | time: 1775518696975 | code: 366 | msg: No historical 

In [21]:
all_rows = []
for req_id, rows in app.data.items():
    all_rows.extend(rows)

df = pd.DataFrame(all_rows)
df["date"] = pd.to_datetime(df["date"])
df = df.sort_values(["symbol", "date"]).reset_index(drop=True)

In [22]:
app.disconnect()
print(app.isConnected())

False


In [23]:
import sys, os
sys.path.insert(0, os.path.abspath(os.path.join(os.getcwd(), "../../..")))

from backtest import run_backtest, BacktestConfig

ASSETS = ["SPY","QQQ","IWM","VXUS","EEM","XLK","XLF","XLV","XLE","VNQ","TLT","GLD"]

close = (
    df.pivot(index="date", columns="symbol", values="close")[ASSETS]
    .sort_index()
    .dropna()
)
print(f"Close DataFrame: {len(close)} rows × {len(close.columns)} assets")
print(f"Date range: {close.index[0].date()} → {close.index[-1].date()}")

config = BacktestConfig(assets=ASSETS)
result = run_backtest(close, config)

result.print_report()
result.summary()

Close DataFrame: 2511 rows × 12 assets
Date range: 2016-04-11 → 2026-04-06

BACKTEST RESULTS
Portfolio               CAGR=+5.97%  Vol=12.79%  Sharpe=0.47  MaxDD=-30.29%
SPY B&H                 CAGR=+13.15%  Vol=18.79%  Sharpe=0.70  MaxDD=-34.10%
Equal-Weight            CAGR=+10.24%  Vol=15.76%  Sharpe=0.65  MaxDD=-31.18%

Regime distribution:
regime
risk-on     31
neutral     24
risk-off    20
Avg effective bets : 21.9


,Strategy,CAGR,Vol,Sharpe,Max DD
0,Portfolio,+5.97%,12.79%,0.47,-30.29%
1,SPY B&H,+13.15%,18.79%,0.70,-34.10%
2,Equal-Weight,+10.24%,15.76%,0.65,-31.18%
